# MVRV-Based Scale Out Exit Strategies

Testing exits based on MVRV valuation levels:

**Logic:** Scale out as MVRV reaches overvaluation zones

- MVRV < 1.0 = Undervalued (buy zone)
- MVRV 1.0-2.0 = Fair value
- MVRV 2.0-2.5 = Getting expensive
- MVRV 2.5-3.0 = Overvalued
- MVRV > 3.0 = Extreme (cycle top zone)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("MVRV-Based Scale Out Exit Strategies 🎯")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df = df[df.index >= '2018-12-15'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")
print(f"MVRV range: {df['mvrv'].min():.2f} to {df['mvrv'].max():.2f}")

In [ ]:
# Entry signals
entry_condition = (
    (df['sopr'] < 1) & 
    (df['sopr_sth'] < 1) & 
    (df['rl_zscore'] > 0.5)
)
entries = entry_condition & ~entry_condition.shift(1).fillna(False)
print(f"Entry signals: {entries.sum()}")

In [ ]:
# Check MVRV at entry points
print("\nMVRV at entry points:")
for date in entries[entries].index:
    print(f"  {date.date()}: MVRV = {df.loc[date, 'mvrv']:.2f}, Price = ${df.loc[date, 'price']:,.0f}")

---
## MVRV Exit Functions

In [ ]:
def exit_simple_trail(df, entry_idx, trail_pct=0.30, stop_loss=0.20):
    """Baseline: Simple trailing stop"""
    price_arr = df['price'].values
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        
        # Trail stop
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
        
        # Stop loss
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price, 'stop_loss'
    
    return len(price_arr) - 1, price_arr[-1], 'end'


def exit_mvrv_hard(df, entry_idx, mvrv_exit=2.5, stop_loss=0.20):
    """Exit 100% when MVRV hits threshold"""
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    entry_price = price_arr[entry_idx]
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        mvrv = mvrv_arr[j]
        
        # MVRV exit
        if mvrv >= mvrv_exit:
            return j, price, f'mvrv>{mvrv_exit}'
        
        # Stop loss
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price, 'stop_loss'
    
    return len(price_arr) - 1, price_arr[-1], 'end'


def exit_mvrv_scale_2(df, entry_idx, mvrv1=2.0, mvrv2=2.5, trail_pct=0.25, stop_loss=0.20):
    """
    Scale out 2 tranches by MVRV:
    - 50% at MVRV > 2.0
    - 50% at MVRV > 2.5 or trail
    """
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    t1_closed = False
    t1_exit = 0.0
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        mvrv = mvrv_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Tranche 1: MVRV > 2.0
        if not t1_closed and mvrv >= mvrv1:
            t1_closed = True
            t1_exit = price
        
        # Tranche 2: MVRV > 2.5
        if t1_closed and mvrv >= mvrv2:
            avg_exit = (t1_exit + price) / 2
            return j, avg_exit, f'mvrv_scale({mvrv1}/{mvrv2})'
        
        # Trail stop for remaining
        if t1_closed and price <= peak * (1 - trail_pct):
            avg_exit = (t1_exit + price) / 2
            return j, avg_exit, 'trail_after_t1'
        
        # Full trail if t1 not hit
        if not t1_closed and price <= peak * (1 - 0.30):
            return j, price, 'trail'
        
        # Stop loss
        if gain <= -stop_loss:
            return j, price, 'stop_loss'
    
    # End of data
    if t1_closed:
        avg_exit = (t1_exit + price_arr[-1]) / 2
        return len(price_arr) - 1, avg_exit, 'end'
    return len(price_arr) - 1, price_arr[-1], 'end'


def exit_mvrv_scale_3(df, entry_idx, mvrv1=2.0, mvrv2=2.5, mvrv3=3.0, trail_pct=0.25, stop_loss=0.20):
    """
    Scale out 3 tranches by MVRV:
    - 33% at MVRV > 2.0
    - 33% at MVRV > 2.5
    - 33% at MVRV > 3.0 or trail
    """
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    t1_closed, t2_closed, t3_closed = False, False, False
    t1_exit, t2_exit, t3_exit = 0.0, 0.0, 0.0
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        mvrv = mvrv_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Tranche 1: MVRV > 2.0
        if not t1_closed and mvrv >= mvrv1:
            t1_closed = True
            t1_exit = price
        
        # Tranche 2: MVRV > 2.5
        if not t2_closed and mvrv >= mvrv2:
            t2_closed = True
            t2_exit = price
        
        # Tranche 3: MVRV > 3.0
        if not t3_closed and mvrv >= mvrv3:
            t3_closed = True
            t3_exit = price
            avg_exit = (t1_exit + t2_exit + t3_exit) / 3
            return j, avg_exit, f'mvrv_full({mvrv1}/{mvrv2}/{mvrv3})'
        
        # Trail stop for remaining position
        if price <= peak * (1 - trail_pct):
            exits = []
            if t1_closed: exits.append(t1_exit)
            if t2_closed: exits.append(t2_exit)
            exits.append(price)  # Remaining at trail
            
            # Weight remaining portions equally
            total_portions = 3
            closed_portions = len([x for x in [t1_closed, t2_closed, t3_closed] if x])
            remaining_portions = total_portions - closed_portions
            
            avg_exit = (sum(exits[:closed_portions]) + price * remaining_portions) / total_portions
            return j, avg_exit, 'trail'
        
        # Stop loss
        if gain <= -stop_loss:
            return j, price, 'stop_loss'
    
    # End of data
    exits = []
    if t1_closed: exits.append(t1_exit)
    if t2_closed: exits.append(t2_exit)
    if t3_closed: exits.append(t3_exit)
    remaining = 3 - len(exits)
    avg_exit = (sum(exits) + price_arr[-1] * remaining) / 3
    return len(price_arr) - 1, avg_exit, 'end'


def exit_mvrv_scale_4(df, entry_idx, mvrv1=1.8, mvrv2=2.2, mvrv3=2.6, mvrv4=3.0, trail_pct=0.25, stop_loss=0.20):
    """
    Scale out 4 tranches by MVRV:
    - 25% at MVRV > 1.8
    - 25% at MVRV > 2.2
    - 25% at MVRV > 2.6
    - 25% at MVRV > 3.0 or trail
    """
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    thresholds = [mvrv1, mvrv2, mvrv3, mvrv4]
    exits = [0.0, 0.0, 0.0, 0.0]
    closed = [False, False, False, False]
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        mvrv = mvrv_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Check tranches
        for i in range(4):
            if not closed[i] and mvrv >= thresholds[i]:
                closed[i] = True
                exits[i] = price
        
        # All tranches closed
        if all(closed):
            avg_exit = sum(exits) / 4
            return j, avg_exit, 'mvrv_full_4'
        
        # Trail stop
        if price <= peak * (1 - trail_pct):
            closed_sum = sum([exits[i] for i in range(4) if closed[i]])
            remaining = sum([1 for c in closed if not c])
            avg_exit = (closed_sum + price * remaining) / 4
            return j, avg_exit, 'trail'
        
        # Stop loss
        if gain <= -stop_loss:
            return j, price, 'stop_loss'
    
    # End of data
    closed_sum = sum([exits[i] for i in range(4) if closed[i]])
    remaining = sum([1 for c in closed if not c])
    avg_exit = (closed_sum + price_arr[-1] * remaining) / 4
    return len(price_arr) - 1, avg_exit, 'end'


def exit_mvrv_trail_activated(df, entry_idx, mvrv_trigger=2.0, trail_pct=0.25, stop_loss=0.20):
    """
    Trail only activates when MVRV > threshold
    """
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    entry_price = price_arr[entry_idx]
    peak = entry_price
    trail_active = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        mvrv = mvrv_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Activate trail when MVRV hits threshold
        if not trail_active and mvrv >= mvrv_trigger:
            trail_active = True
        
        # Trail stop
        if trail_active and price <= peak * (1 - trail_pct):
            return j, price, f'mvrv_trail({mvrv_trigger})'
        
        # Stop loss
        if gain <= -stop_loss:
            return j, price, 'stop_loss'
    
    return len(price_arr) - 1, price_arr[-1], 'end'

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, fees=0.001, **kwargs):
    """Run backtest with given exit function"""
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price, exit_reason = exit_func(df, entry_idx, **kwargs)
        
        entry_price = df['price'].iloc[entry_idx]
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fees)
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'entry_mvrv': df['mvrv'].iloc[entry_idx],
            'exit_mvrv': df['mvrv'].iloc[exit_idx],
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    
    return trades_df


def calc_metrics(trades, initial_capital, df):
    """Calculate performance metrics"""
    if len(trades) == 0:
        return {'total_return': 0, 'sharpe': 0, 'max_dd': 0, 'win_rate': 0}
    
    final_equity = trades['equity'].iloc[-1]
    total_return = (final_equity / initial_capital) - 1
    
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    
    win_rate = (trades['net_return'] > 0).mean()
    
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 else 0
    
    # Max drawdown
    equity = [initial_capital] + list(trades['equity'])
    peak = equity[0]
    max_dd = 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    
    # Buy & hold
    bh_return = (df['price'].iloc[-1] / df.loc[trades['entry_date'].iloc[0], 'price']) - 1
    
    # Profit factor
    winners = trades[trades['net_return'] > 0]['net_return'].sum()
    losers = abs(trades[trades['net_return'] <= 0]['net_return'].sum())
    pf = winners / losers if losers > 0 else float('inf')
    
    return {
        'total_return': total_return,
        'cagr': cagr,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'profit_factor': pf,
        'n_trades': len(trades),
        'bh_return': bh_return,
        'final_equity': final_equity
    }

---
## Run All MVRV Exit Strategies

In [ ]:
INITIAL = 100000

strategies = [
    # Baseline
    ('1. Simple 30% Trail (baseline)', exit_simple_trail, {'trail_pct': 0.30, 'stop_loss': 0.20}),
    
    # Hard MVRV exits
    ('2. MVRV > 2.0 Hard Exit', exit_mvrv_hard, {'mvrv_exit': 2.0, 'stop_loss': 0.20}),
    ('3. MVRV > 2.5 Hard Exit', exit_mvrv_hard, {'mvrv_exit': 2.5, 'stop_loss': 0.20}),
    ('4. MVRV > 3.0 Hard Exit', exit_mvrv_hard, {'mvrv_exit': 3.0, 'stop_loss': 0.20}),
    
    # MVRV-triggered trail
    ('5. MVRV > 2.0 → 25% Trail', exit_mvrv_trail_activated, {'mvrv_trigger': 2.0, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    ('6. MVRV > 2.0 → 20% Trail', exit_mvrv_trail_activated, {'mvrv_trigger': 2.0, 'trail_pct': 0.20, 'stop_loss': 0.20}),
    ('7. MVRV > 2.5 → 25% Trail', exit_mvrv_trail_activated, {'mvrv_trigger': 2.5, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    
    # MVRV Scale Out 2 tranches
    ('8. Scale 2x: MVRV 2.0/2.5', exit_mvrv_scale_2, {'mvrv1': 2.0, 'mvrv2': 2.5, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    ('9. Scale 2x: MVRV 2.0/3.0', exit_mvrv_scale_2, {'mvrv1': 2.0, 'mvrv2': 3.0, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    ('10. Scale 2x: MVRV 2.2/2.8', exit_mvrv_scale_2, {'mvrv1': 2.2, 'mvrv2': 2.8, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    
    # MVRV Scale Out 3 tranches
    ('11. Scale 3x: MVRV 2.0/2.5/3.0', exit_mvrv_scale_3, {'mvrv1': 2.0, 'mvrv2': 2.5, 'mvrv3': 3.0, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    ('12. Scale 3x: MVRV 1.8/2.3/2.8', exit_mvrv_scale_3, {'mvrv1': 1.8, 'mvrv2': 2.3, 'mvrv3': 2.8, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    ('13. Scale 3x: MVRV 2.0/2.5/3.5', exit_mvrv_scale_3, {'mvrv1': 2.0, 'mvrv2': 2.5, 'mvrv3': 3.5, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    
    # MVRV Scale Out 4 tranches
    ('14. Scale 4x: MVRV 1.8/2.2/2.6/3.0', exit_mvrv_scale_4, {'mvrv1': 1.8, 'mvrv2': 2.2, 'mvrv3': 2.6, 'mvrv4': 3.0, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    ('15. Scale 4x: MVRV 2.0/2.4/2.8/3.2', exit_mvrv_scale_4, {'mvrv1': 2.0, 'mvrv2': 2.4, 'mvrv3': 2.8, 'mvrv4': 3.2, 'trail_pct': 0.25, 'stop_loss': 0.20}),
]

results = []

print("MVRV-BASED EXIT STRATEGY COMPARISON")
print("="*150)
print(f"{'Strategy':<40} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'Win%':>8} {'PF':>8} {'MaxDD':>10} {'Trades':>8} {'Final $':>14}")
print("-"*150)

for name, func, kwargs in strategies:
    trades = run_backtest(df, entries, func, INITIAL, **kwargs)
    m = calc_metrics(trades, INITIAL, df)
    
    print(f"{name:<40} {m['total_return']*100:>+11.0f}% {m['cagr']*100:>+9.1f}% "
          f"{m['sharpe']:>8.2f} {m['win_rate']*100:>7.0f}% {m['profit_factor']:>8.2f} "
          f"{m['max_dd']*100:>9.0f}% {m['n_trades']:>8} ${m['final_equity']:>13,.0f}")
    
    results.append({'name': name, 'trades': trades, 'metrics': m})

print("-"*150)
print(f"{'Buy & Hold':<40} {m['bh_return']*100:>+11.0f}%")

In [ ]:
# Rankings
print("\n" + "="*60)
print("RANKINGS")
print("="*60)

# Best by return
by_return = sorted(results, key=lambda x: x['metrics']['total_return'], reverse=True)
print(f"\n📈 Best by Total Return:")
for i, r in enumerate(by_return[:5]):
    print(f"   {i+1}. {r['name']}: {r['metrics']['total_return']*100:+,.0f}%")

# Best by Sharpe
by_sharpe = sorted(results, key=lambda x: x['metrics']['sharpe'], reverse=True)
print(f"\n📊 Best by Sharpe Ratio:")
for i, r in enumerate(by_sharpe[:5]):
    print(f"   {i+1}. {r['name']}: {r['metrics']['sharpe']:.2f}")

# Best by Win Rate
by_wr = sorted(results, key=lambda x: x['metrics']['win_rate'], reverse=True)
print(f"\n🎯 Best by Win Rate:")
for i, r in enumerate(by_wr[:5]):
    print(f"   {i+1}. {r['name']}: {r['metrics']['win_rate']*100:.0f}%")

In [ ]:
# Trade details for top strategies
print("\n" + "="*120)
print("TRADE DETAILS")
print("="*120)

for r in by_return[:3]:
    print(f"\n{r['name']}")
    print("-"*100)
    trades = r['trades']
    for _, t in trades.iterrows():
        print(f"  {t['entry_date'].date()} → {t['exit_date'].date()}: "
              f"${t['entry_price']:,.0f} → ${t['exit_price']:,.0f} = {t['net_return']*100:+.0f}% "
              f"| MVRV: {t['entry_mvrv']:.2f} → {t['exit_mvrv']:.2f} "
              f"| {t['exit_reason']}")

In [ ]:
# Score comparison
for r in results:
    m = r['metrics']
    r['score'] = (m['total_return'] * m['sharpe']) / abs(m['max_dd']) if m['max_dd'] != 0 else 0

by_score = sorted(results, key=lambda x: x['score'], reverse=True)

print("\n" + "="*60)
print("OVERALL RANKING (Return × Sharpe / MaxDD)")
print("="*60)
for i, r in enumerate(by_score[:5]):
    m = r['metrics']
    print(f"\n{i+1}. {r['name']}")
    print(f"   Return: {m['total_return']*100:+,.0f}%")
    print(f"   Sharpe: {m['sharpe']:.2f}")
    print(f"   Max DD: {m['max_dd']*100:.0f}%")
    print(f"   Win Rate: {m['win_rate']*100:.0f}%")
    print(f"   Score: {r['score']:.2f}")

In [ ]:
# Visualization
import plotly.graph_objects as go

names = [r['name'].split('. ')[1] if '. ' in r['name'] else r['name'] for r in results]
returns = [r['metrics']['total_return'] * 100 for r in results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=names,
    y=returns,
    marker_color=['green' if 'Simple' in n else 'blue' for n in names],
    text=[f"{r:+,.0f}%" for r in returns],
    textposition='outside'
))

fig.update_layout(
    title='MVRV Exit Strategy Comparison',
    yaxis_title='Total Return %',
    height=600,
    xaxis_tickangle=-45
)
fig.show()